<a href="https://colab.research.google.com/github/CompOmics/lsabd-machine-learning-tutorials/blob/main/notebooks/2-overfitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overfitting and regularization

This notebook demonstrates how machine learning models can achieve 100% training accuracy but perform poorly on test data due to overfitting. We'll visualize this phenomenon using a synthetic non-linear dataset (half-moons) and a simple decision tree. Then, we'll explore how regularization techniques like limiting tree depth can help mitigate overfitting. We will cover:

1. The bias-variance trade-off
2. Overfitting with an unrestricted decision tree
3. Regularizing a decision tree by limiting its depth
4. Visualizing decision boundaries and the effect of tree depth
5. Regularizing a logistic regression model
6. Regularizing a linear regression model

Note that throughout the notebook, there are links to the relevant documentation for the functions and classes used. Click on the links and explore the documentation to deepen your understanding.

Overfitting occurs when a model learns the training data too well, capturing noise and outliers rather than just the underlying pattern. This results in:
- Excellent performance on training data
- Poor performance on new, unseen data (test data)

In biological datasets, especially -omics data (e.g., transcriptomics, proteomics), overfitting is a common challenge due to the high dimensionality and complexity of the data. We often get a low number of samples (10 to 100) compared to a very high number of features (thousands to tens of thousands). This imbalance makes it easy for models to memorize the training data instead of learning generalizable patterns. This is often called the "curse of dimensionality."

The amount of data matters as well. With few samples, even a simple model can memorize the training data, while with more samples the noise averages out and individual points are harder to memorize. More data therefore helps against overfitting, but not against underfitting: a model that is too simple stays too simple, no matter how much data it sees.

Decision trees are particularly prone to overfitting because they can create very complex decision boundaries that perfectly separate the training data. Nevertheless, overfitting can occur with virtually any machine learning algorithm if not properly regularized. Overfitting is closely related to the bias-variance trade-off, which we introduce first.

First, let's import the necessary libraries:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set the style for our visualizations
sns.set_style("whitegrid")
sns.set_palette("deep")

# Set random seed for reproducibility
np.random.seed(42)

## The bias-variance trade-off

Overfitting is closely related to the bias-variance trade-off. The error a model makes on unseen data can be split into three parts: bias, variance, and irreducible noise in the data itself. The first two depend on the model:

- **Bias** is the systematic error of a model that is too simple to capture the true pattern. A high-bias model is consistently off target, regardless of the training data it sees. This is **underfitting**.
- **Variance** is the error from a model that is too sensitive to the specific training data, including its noise. A high-variance model gives very different predictions when trained on different samples. This is **overfitting**.

The classic way to visualize this is a bullseye. Imagine training the same type of model many times on different training samples. Each dot below is the prediction of one such model, and the center of the target is the true value:

In [ ]:
# Each dot is the prediction of a model trained on a different training sample
rng = np.random.default_rng(1)
settings = [
    ("Low bias, low variance", 0.0, 0.3),
    ("High bias, low variance", 2.0, 0.3),
    ("Low bias, high variance", 0.0, 1.0),
    ("High bias, high variance", 2.0, 1.0),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, bias, spread) in zip(axes, settings):
    # Draw the target as concentric rings
    for radius in (4, 3, 2, 1):
        ring_color = "lightgrey" if radius % 2 == 0 else "white"
        ax.add_patch(plt.Circle((0, 0), radius, facecolor=ring_color, edgecolor="grey"))
    # Draw the predictions: bias shifts them away from the center, variance spreads them out
    predictions = rng.normal(loc=[0.8 * bias, 0.6 * bias], scale=spread, size=(15, 2))
    ax.scatter(predictions[:, 0], predictions[:, 1], color="C3", zorder=3)
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), aspect="equal", xticks=[], yticks=[])
    ax.set_title(title)
    ax.grid(False)
plt.show()

Simple models tend to have high bias and low variance, while complex models tend to have low bias and high variance. Increasing model complexity lowers the bias but raises the variance, so the best performance on unseen data is found at an intermediate complexity. Regularization is the tool to control this trade-off: it deliberately introduces a little bias to reduce the variance. Throughout this notebook, we will see this play out for decision trees and linear models.

*Question: In which of the four panels would you place an overfitted model? And an underfitted model?*

Write your answer here:

...

## Visualizing overfitting with a decision tree model

### Data generation

We'll generate a synthetic non-linear dataset using scikit-learn's [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function. This creates two interleaving half-moons, which is a classic example of data that requires non-linear decision boundaries.

To make overfitting more pronounced, we'll also add some noise and outliers by flipping the labels of a few random points.

In [ ]:
from sklearn.datasets import make_moons

# Generate a non-linear dataset (half-moons)
features, targets = make_moons(n_samples=500, noise=0.3, random_state=1)

# Add some noise/outliers to make overfitting more pronounced
# Add 20 random outliers
n_outliers = 20
outlier_indices = np.random.choice(len(features), n_outliers, replace=False)

# Flip the labels for these outliers
targets[outlier_indices] = 1 - targets[outlier_indices]

# Visualize the dataset
sns.scatterplot(x=features[:, 0], y=features[:, 1], hue=targets, alpha=0.7)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

*Question: How many features does this dataset have?*

Write your answer here:

...

### Data splitting

We'll split our data into training (70%) and testing (30%) sets. The training set will be used to train our models, while the testing set will be used to evaluate their performance on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
train_features, test_features, train_targets, test_targets = train_test_split(features, targets, test_size=0.3, random_state=42)

print(f"Training set size: {train_features.shape[0]} samples")
print(f"Testing set size: {test_features.shape[0]} samples")

### Training an unrestricted decision tree

First, let's train a decision tree with default parameters. By default, scikit-learn's [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/tree.html#classification) will grow a tree until all leaves are pure (contain samples of only one class) or until all leaves contain less than `min_samples_split` samples.

This allows the tree to potentially grow very deep and create a complex decision boundary that perfectly fits the training data, including noise and outliers.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# Create a decision tree classifier with default parameters
# This allows tree to grow fully and potentially overfit
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(train_features, train_targets)

# Make predictions to evaluate performance
train_predictions = classifier.predict(train_features)
test_predictions = classifier.predict(test_features)

# Calculate accuracies
train_accuracy = accuracy_score(train_targets, train_predictions)
test_accuracy = accuracy_score(test_targets, test_predictions)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Testing accuracy: {test_accuracy:.4f}")

*Question: Do you already observe something interesting about the accuracies?*

Write your answer here:

...

### Training a regularized decision tree

Now, let's train a regularized decision tree by limiting its maximum depth to 3. This prevents the tree from creating an overly complex decision boundary and forces it to focus on the most important patterns in the data.

In [ ]:
# Create a regularized tree with limited depth for comparison
reg_classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
reg_classifier.fit(train_features, train_targets)

reg_train_predictions = reg_classifier.predict(train_features)
reg_test_predictions = reg_classifier.predict(test_features)

reg_train_accuracy = accuracy_score(train_targets, reg_train_predictions)
reg_test_accuracy = accuracy_score(test_targets, reg_test_predictions)

print("Regularized tree:")
print(f"Training accuracy: {reg_train_accuracy:.4f}")
print(f"Testing accuracy: {reg_test_accuracy:.4f}")

*Question: How do the accuracies compare to the unrestricted tree? What does that tell you?*

Write your answer here:

...

### Visualizing decision boundaries

Let's visualize the decision boundaries of our models using the `mlxtend` library. This will help us see how the unrestricted tree creates a complex boundary that perfectly fits the training data, while the regularized tree creates a simpler boundary that generalizes better to unseen data.

In [ ]:
from mlxtend.plotting import plot_decision_regions

# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot decision boundary for overfitted tree
plot_decision_regions(train_features, train_targets, clf=classifier, ax=ax1)
ax1.set_title(
    f"Overfitted Tree (depth={classifier.get_depth()})\n"
    f"Train: {train_accuracy:.2f}, Test: {test_accuracy:.2f}",
)

# Plot decision boundary for regularized tree
plot_decision_regions(train_features, train_targets, clf=reg_classifier, ax=ax2)
ax2.set_title(
    f"Regularized Tree (depth={reg_classifier.get_depth()})\n"
    f"Train: {reg_train_accuracy:.2f}, Test: {reg_test_accuracy:.2f}"
)

for ax in (ax1, ax2):
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

fig.tight_layout()
plt.show()

Investigate the plots to see how the decision boundaries differ between the unrestricted and regularized trees. Notice how the unrestricted tree's boundary is highly irregular, while the regularized tree's boundary better captures the overall structure of the data.

*Question: Where did the unregularized tree overfit the data? What do you notice about the irregularly fitted regions?*

Write your answer here:

...

### The effect of tree depth on overfitting

Let's explore how the maximum depth of the tree affects its performance on training and testing data. We'll train multiple trees with different maximum depths and plot their accuracies.

In [ ]:
# Parameter values to try
max_depths = range(1, 21)
train_scores = []
test_scores = []

for depth in max_depths:
    # Train tree with specific depth
    classifier = DecisionTreeClassifier(max_depth=depth, random_state=42)
    classifier.fit(train_features, train_targets)

    # Record accuracies
    train_scores.append(accuracy_score(train_targets, classifier.predict(train_features)))
    test_scores.append(accuracy_score(test_targets, classifier.predict(test_features)))

# Plot accuracies vs tree depth
c1, c2 = sns.color_palette("deep", 2)
plt.figure(figsize=(10, 6))
plt.plot(max_depths, train_scores, "o-", color=c1, label="Training accuracy")
plt.plot(max_depths, test_scores, "o-", color=c2, label="Testing accuracy")
plt.axhline(y=1.0, color="grey", linestyle=":", label="Perfect accuracy")
plt.xticks(range(0, 21, 2))
plt.xlabel("Max Tree Depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Performance vs Tree Depth")
plt.legend()
plt.show()

This resulting figure is a form of a "learning curve," showing how model complexity (in this case, tree depth) impacts performance. As the tree depth increases, the training accuracy approaches 100%, but the test accuracy may decrease after a certain point due to overfitting. In terms of the bias-variance trade-off: very shallow trees have a high bias (underfitting), while very deep trees have a high variance (overfitting).

*Question: What is the ideal value of max depth for this dataset?*

Write your answer here:

...

## Regularization for linear models

Linear models, such as linear regression and logistic regression, can also suffer from overfitting. For these models, we instead apply regularization techniques like Lasso (L1) and Ridge (L2) regularization. These techniques add a penalty term to the loss function that discourages overly complex models by penalizing large weights.

### Regularizing a logistic regression model

To regularize a [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) model in scikit-learn, you can set the `penalty` parameter to `'l1'` for Lasso or `'l2'` for Ridge, and adjust the `C` parameter (inverse of regularization strength).

First, we generate a synthetic classification dataset with many features (200) relative to the number of samples (300), and some label noise. This makes it prone to overfitting:

In [ ]:
from sklearn.datasets import make_classification

# Generate a classification dataset with more features and more noise
features, targets = make_classification(
    n_samples=300,
    n_features=200,
    n_informative=50,
    flip_y=0.2,
    shift=0.1,
    random_state=42,
)

# Split the dataset into training and testing sets
train_features, test_features, train_targets, test_targets = train_test_split(features, targets, test_size=0.3, random_state=42)

Now we can fit both an unregularized and a regularized logistic regression model to this dataset and compare their performances.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic regression without regularization
log_reg = LogisticRegression(penalty=None)
log_reg.fit(train_features, train_targets)

# Predictions and accuracy
train_pred = log_reg.predict(train_features)
test_pred = log_reg.predict(test_features)

print("Logistic Regression without regularization")
print(f"Training accuracy: {accuracy_score(train_targets, train_pred):.4f}")
print(f"Testing accuracy: {accuracy_score(test_targets, test_pred):.4f}")
print()

# Logistic regression with L2 regularization
log_reg_reg = LogisticRegression(penalty='l2', C=0.001, max_iter=1000)
log_reg_reg.fit(train_features, train_targets)
# Predictions and accuracy
train_pred_reg = log_reg_reg.predict(train_features)
test_pred_reg = log_reg_reg.predict(test_features)
print("Logistic Regression with L2 regularization")
print(f"Training accuracy: {accuracy_score(train_targets, train_pred_reg):.4f}")
print(f"Testing accuracy: {accuracy_score(test_targets, test_pred_reg):.4f}")

*Question: What difference do you see between the regularized and unregularized logistic regression models?*

Write your answer here:

...

Let's compare the performance of logistic regression models with varying levels of regularization. This can be tuned with the `C` parameter in `LogisticRegression`, where smaller values specify stronger regularization.

In [ ]:
# Learning curves for regularization levels:
c_values = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
train_accuracies = []
test_accuracies = []

for C in c_values:
    log_reg = LogisticRegression(penalty='l2', C=C, max_iter=1000)
    log_reg.fit(train_features, train_targets)
    train_accuracies.append(accuracy_score(train_targets, log_reg.predict(train_features)))
    test_accuracies.append(accuracy_score(test_targets, log_reg.predict(test_features)))

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(c_values, train_accuracies, "o-", label="Training accuracy")
plt.plot(c_values, test_accuracies, "o-", label="Testing accuracy")
plt.xscale("log")
plt.xlabel("Regularization Strength (C)")
plt.ylabel("Accuracy")
plt.title("Logistic Regression Performance vs Regularization Strength")
plt.legend()
plt.show()


*Question: What value of C would you choose in this case?*

Write your answer here:

...

### Regularizing a linear regression model

To regularize a [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model, you can use the [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) classes. For these models, the regularization strength is controlled by the `alpha` parameter, in the range of [0, ∞), where higher values specify stronger regularization (opposite of `C` in logistic regression).

Let's create a synthetic regression dataset that is prone to overfitting by having a small number of samples, a large number of features, and a high noise level:

In [ ]:
from sklearn.datasets import make_regression

features, targets = make_regression(
    n_samples=80,         # Small sample size
    n_features=100,       # Many features
    n_informative=3,      # Only some truly useful features
    noise=80,             # High noise level
    random_state=42
)
train_features, test_features, train_targets, test_targets = train_test_split(features, targets, test_size=0.3, random_state=42)

And train Ridge regression models with varying levels of regularization:

In [ ]:
from sklearn.linear_model import Ridge

# Learning curves for regularization levels:
alpha_values = [0.1, 1, 10, 100, 1000]
train_r2_scores = []
test_r2_scores = []
for alpha in alpha_values:
    ridge_reg = Ridge(alpha=alpha)
    ridge_reg.fit(train_features, train_targets)
    train_r2_scores.append(ridge_reg.score(train_features, train_targets))
    test_r2_scores.append(ridge_reg.score(test_features, test_targets))

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(alpha_values, train_r2_scores, label="Training R²", marker="o")
plt.plot(alpha_values, test_r2_scores, label="Testing R²", marker="o")
plt.xscale("log")
plt.xlabel("Regularization Strength (alpha)")
plt.ylabel("R² Score")
plt.legend()
plt.show()

*Question: What is the best value of alpha for this dataset?*

Write your answer here:

...

*Question: What happens towards the left of the plot (low alpha values) and towards the right of the plot (high alpha values)? In other words, where is the model overfitting, and where is it underfitting?*

Write your answer here:

...

## Summarized

- Overfitting occurs when a model learns the training data too well, capturing noise and outliers, leading to poor generalization on unseen data.
- Decision trees are particularly prone to overfitting, but regularization techniques like limiting tree depth can help mitigate this.
- Linear models can also overfit, but regularization techniques like Lasso (L1) and Ridge (L2) can help control model complexity.
- Too little regularization leads to overfitting, while too much leads to underfitting. The right amount is found by comparing training and test performance.
- Overfitting is more likely with few samples and many features. More data helps against overfitting, but not against underfitting.
- Overfitting corresponds to a model with high variance, and underfitting to a model with high bias. Regularization trades a little bias for a lower variance, to find the best balance (the bias-variance trade-off).